In [ ]:
import os
import s3fs

In [ ]:
from pyspark.sql import SparkSession
from pipeline_builder import create_pipeline

### Configuration Spark

In [ ]:
spark = (SparkSession.builder
        .appName("SparkExample")
        .config("spark.executor.memory", "4g") # 2g default
        .config("spark.driver.memory", "4g") # 2g default
        # .config("spark.executor.instances", 10)
        .getOrCreate())

In [ ]:
# spark.stop()

In [ ]:
spark.sparkContext.getConf().get("spark.executor.memory")

In [ ]:
for conf in spark.sparkContext.getConf().getAll():
    print(conf)

In [ ]:
# BUCKET = "fgao-ensae"
# FILE_KEY_S3 = "Data_spark/ML_Lib"
# s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}/"
# print(s3_path)

BUCKET = 'ematzner-ensae'
FILE_KEY_S3 = 'Mllib'
s3_path = f's3a://{BUCKET}/{FILE_KEY_S3}'
s3_path

In [ ]:
df_ini = spark.read.csv(s3_path+"Cost_of_Living_Index_2022.csv",
                    header= True,
                    inferSchema=True
                   )

In [ ]:
df_ini.show()

In [ ]:
# df_ini = df_ini.withColumnRenamed('Cost of Living Index', label)

In [ ]:
label = 'Cost of Living Index'

In [ ]:
df = df_ini.drop('Rank', 'Country')

In [ ]:
df.show(2)

In [ ]:
from pipeline_builder import create_pipeline
preproc, label_col, task_type = create_pipeline(df, label=label)

In [ ]:
# from pyspark.ml.feature import VectorAssembler

# df_ini.columns[3:]

# fa = VectorAssembler(
#     inputCols=df_ini.columns[3:],
#     outputCol='features'
# )

# df = (fa.transform(df_ini)
#       .select('features', 'Cost of Living Index')
#       .withColumnRenamed('Cost of Living Index', label)
#      )

# df.show(2, False)

In [ ]:
for stage in preproc.getStages():
    print(stage)

In [ ]:
train, test = df.randomSplit([.8, .2], seed = 42)
print(train.count())
print(test.count())

# test = test.fillna(0, subset=numeric_cols)

In [ ]:
preproc_fitted = preproc.fit(train)

In [ ]:
test_transformed = preproc_fitted.transform(test)
test_transformed.show()

In [ ]:
test_transformed.select(['features', label]).show(10, False)

# Linear Regression
## 1. Defaut Pipeline

In [ ]:
from pyspark.ml.regression import LinearRegression
# from pyspark.ml.feature import StandardScaler
from pyspark.ml.pipeline import Pipeline

In [ ]:
lr = LinearRegression(
    featuresCol='features',
    labelCol= label
)

In [ ]:
pip = Pipeline(stages= [preproc, lr])

In [ ]:
pip_fitted = pip.fit(train)
predict = pip_fitted.transform(test)
predict.show()

## 2. Model Evaluation

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
evaluator = RegressionEvaluator(
    metricName= 'rmse',
    predictionCol='prediction',
    labelCol=label
)

In [ ]:
evaluator.evaluate(predict)

## 3. Tuning

In [ ]:
print(lr.explainParams())

Dans Spark, lorsque tu utilises un modèle de régression linéaire, comme LinearRegression, il y a deux hyperparamètres importants à comprendre : elasticNetParam et regParam. Voici une explication rapide de chacun et de leur rôle dans la régularisation.

1. **regParam (paramètre de régularisation : lambda)**  :
Contrôle l'intensité de la régularisation, c'est-à-dire à quel point tu veux pénaliser les coefficients du modèle. Plus la valeur de regParam est grande, plus les coefficients seront réduits pour limiter la complexité du modèle.

But : Limiter le sur-apprentissage en ajoutant une pénalisation aux coefficients du modèle pour éviter des valeurs trop grandes.
Il est utilisé à la fois pour la régularisation de type L1 (Lasso) et la régularisation de type L2 (Ridge). La valeur que tu fixes ici ajuste globalement le poids de la régularisation. Lorsque regParam est plus grand, cela augmente l'intensité de la pénalisation, ce qui pousse les coefficients de ton modèle à devenir plus petits.Cela contraint les coefficients (w_i) à être proches de zéro pour minimiser le terme de pénalisation. En pratique, cela empêche les coefficients de prendre des valeurs élevées, ce qui rend le modèle plus simple et moins flexible.

Si regParam est proche de zéro ou égal à zéro, il n'y a presque aucune pénalisation sur les coefficients. Dans ce cas, les coefficients peuvent être grands pour mieux s'adapter aux données, mais cela peut conduire à du surapprentissage (overfitting), car le modèle peut devenir trop complexe.

2. **elasticNetParam (paramètre Elastic Net, alpha)** :
Définition : Détermine le type de régularisation, c'est-à-dire l'équilibre entre la régularisation L1 (Lasso) et L2 (Ridge). Ce paramètre détermine l'équilibre entre L1 (Lasso) et L2 (Ridge) dans la régularisation.
elasticNetParam = 0 : Modèle de régularisation purement Ridge (L2).
elasticNetParam = 1 : Modèle de régularisation purement Lasso (L1).
Entre 0 et 1 : Mélange de régularisation L1 et L2, appelé Elastic Net.
Relation entre les deux :
regParam : Fixe l'intensité globale de la régularisation.
elasticNetParam : Contrôle le type de régularisation (L1 vs L2 ou un mélange des deux).

En résumé :

* regParam détermine à quel point tu veux régulariser ton modèle, indépendamment du type de régularisation.
* elasticNetParam détermine la proportion de régularisation L1 (Lasso) et L2 (Ridge) utilisée dans le modèle.


Exemple d'utilisation dans PySpark :

from pyspark.ml.regression import LinearRegression

lr = LinearRegression(regParam=0.1, elasticNetParam=0.5)
Dans cet exemple, tu as une régularisation avec 50% de L1 et 50% de L2, et la force de cette régularisation est contrôlée par regParam=0.1.

**maxIter**

Le paramètre maxIter dans la régression linéaire de Spark (et dans d'autres algorithmes d'apprentissage machine) représente le nombre maximum d'itérations que l'algorithme d'optimisation est autorisé à effectuer pour converger vers une solution optimale.

**Rôle** : Lorsque Spark utilise un algorithme d'optimisation, comme la descente de gradient, pour minimiser la fonction de coût (erreur), il effectue une série d'itérations pour ajuster les coefficients du modèle. Chaque itération met à jour les coefficients en fonction de l'algorithme choisi et des données.
**Limite imposée** : maxIter impose une limite au nombre de ces itérations. Si l'algorithme n'a pas convergé (c'est-à-dire trouvé les coefficients optimaux) avant d'atteindre ce nombre d'itérations, il s'arrêtera et retournera les coefficients trouvés jusqu'à ce point.

**Regression lineaire n'est pas un algorithme de Gradient Descent ? pour quoi on a besoin de faire ces itérations ?**

La régression linéaire elle-même n'est pas spécifiquement liée à l'algorithme de Gradient Descent (GD), mais pour entraîner un modèle de régression linéaire sur de grandes quantités de données, comme c'est souvent le cas avec Apache Spark et MLlib, des méthodes d'optimisation itératives comme le Gradient Descent ou L-BFGS (Limited-memory Broyden-Fletcher-Goldfarb-Shanno) sont souvent utilisées.

Pourquoi utiliser des itérations en régression linéaire ?

Régression linéaire exacte (résolution analytique) : Théoriquement, la régression linéaire peut être résolue en une seule étape en utilisant des formules analytiques, comme l'inversion de matrice (la pseudo-inverse de la matrice des features). Cependant, cette méthode n'est pas toujours pratique pour des données volumineuses, car elle est coûteuse en calculs et en mémoire, surtout lorsque le nombre de features (variables) est grand. L'inversion d'une matrice a une complexité de O(n3), ce qui devient inenvisageable pour de très grandes matrices.

Méthodes itératives (comme Gradient Descent) : Pour cette raison, dans des systèmes distribués comme Spark, on préfère souvent utiliser des méthodes itératives. Le Gradient Descent est l'un des algorithmes d'optimisation les plus populaires pour ajuster les poids d'un modèle en minimisant la fonction de coût (erreur quadratique moyenne, par exemple). Il fonctionne par petites étapes à chaque itération pour ajuster les poids dans la direction de la pente descendante de la fonction de coût.
Pourquoi plusieurs itérations ?

Gradient Descent : À chaque itération, l'algorithme met à jour les poids du modèle pour réduire l'erreur. Mais une seule mise à jour ne suffit pas. Il faut généralement plusieurs itérations pour que l'algorithme converge vers une solution optimale (ou proche de l'optimal).

Précision et Convergence : Le nombre d'itérations est un compromis entre le temps d'exécution et la précision. Si vous limitez trop les itérations, l'algorithme pourrait s'arrêter avant d'avoir convergé vers une solution optimale. Si vous en faites trop, vous pourriez passer du temps à améliorer une solution qui est déjà "assez bonne".

Algorithme de L-BFGS : Dans Spark MLlib, la régression linéaire peut également utiliser une variante du Gradient Descent appelée L-BFGS. L-BFGS est un algorithme quasi-Newtonien plus rapide que le Gradient Descent traditionnel. Cependant, il a également besoin d'itérations pour affiner progressivement les poids du modèle jusqu'à ce qu'il atteigne un minimum local de la fonction de coût.

**En résumé**

Même si la régression linéaire est, en théorie, solvable par des méthodes analytiques, les méthodes itératives comme le Gradient Descent sont plus pratiques pour les grandes quantités de données, car elles sont plus efficaces en termes de calcul et de mémoire. Les itérations sont nécessaires pour permettre à l'algorithme d'optimisation de converger vers une solution optimale ou acceptable.

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
params = (ParamGridBuilder()
          .addGrid(lr.regParam, [0, .01, .1, 1]) # Lambda
          .addGrid(lr.elasticNetParam, [0, .5, 1]) # alpha
          .addGrid(lr.maxIter, [100, 200, 500])
          .build()
    
)

In [ ]:
cv = CrossValidator(
    estimator=pip,
    estimatorParamMaps=params,
    evaluator=evaluator,
    numFolds= 5
)

In [ ]:
cv_fitted = cv.fit(train)

In [ ]:
best_pip = cv_fitted.bestModel
type(best_pip)

In [ ]:
import numpy as np
np.argmin(cv_fitted.avgMetrics) # 0 (position)
# cv_fitted.avgMetrics

In [ ]:
cv_fitted.getEstimatorParamMaps()

print(cv_fitted.getEstimatorParamMaps()[np.argmin(cv_fitted.avgMetrics)])

# OLS, pas de pénalité

In [ ]:
# print(best_pip.stages[-1]._java_obj.parent().getRegParam())
# print(best_pip.stages[-1]._java_obj.parent().getElasticNetParam())
# print(best_pip.stages[-1]._java_obj.parent().getMaxIter())

In [ ]:
evaluator.evaluate(best_pip.transform(test))

## Model Persisting

In [ ]:
s3_path

In [ ]:
best_pip.write().overwrite().save(s3_path+'/best_pip_model')

# best_pip.write().overwrite().save('/FileStore/tables/best_pip_model')

# dbutils.fs.ls("dbfs:/FileStore/tables")

In [ ]:
from pyspark.ml.pipeline import PipelineModel

In [ ]:
pip_loaded = PipelineModel.load(s3_path+'/best_pip_model')

# pip_persisted = PipelineModel.load("dbfs:/FileStore/tables/best_pip_model")

In [ ]:
pip_loaded.transform(test).show()

In [ ]:
df

In [ ]:
preproc, label_col, task_type = create_pipeline(df, label=label)

# Fit pipeline
# df_prepared = pipeline.fit(df).transform(df)

In [ ]:
task_type

In [ ]:
preproc.getStages()